# 01 · Dataset exploration

This notebook inspects the deterministic synthetic datasets that ship with the
Finora. They are generated with a fixed seed (42) by
`datasets/generate_datasets.py`, so the project is reproducible without using
any personal or scraped data.

Files used by the application:

| File | Purpose |
| --- | --- |
| `datasets/expense_transactions.csv` | Classifier training and historical expense analysis |
| `datasets/monthly_expenses.csv` | Monthly aggregate forecasting dataset |
| `datasets/budget_data.csv` | Example category budgets |
| `datasets/financial_insights.csv` | Insight-rule templates |

> Nothing here is hardcoded into the running app. The dashboard, budgets,
> insights and forecasts are computed from the authenticated user's database
> records at request time.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "ml"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")


## Transactions dataset

In [ ]:
transactions = pd.read_csv(PROJECT_ROOT / "datasets" / "expense_transactions.csv")
transactions["date"] = pd.to_datetime(transactions["date"])
print(f"Rows: {len(transactions):,}")
print(f"Date range: {transactions['date'].min():%Y-%m-%d} to {transactions['date'].max():%Y-%m-%d}")
transactions.head()


In [ ]:
transactions.dtypes


## Class balance

The classifier is trained only on expense rows across eight categories.

In [ ]:
expenses = transactions[transactions["transaction_type"] == "Expense"].copy()
category_counts = expenses["category"].value_counts()
category_counts


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
category_counts.sort_values().plot(kind="barh", color="#3B82F6", ax=ax)
ax.set_title("Expense transactions by category")
ax.set_xlabel("Transactions")
ax.set_ylabel("")
fig.tight_layout()
plt.show()


## Monthly spending trend

In [ ]:
monthly = (
    transactions.set_index("date")
    .groupby([pd.Grouper(freq="ME"), "transaction_type"])["amount"]
    .sum()
    .unstack(fill_value=0)
)
monthly.tail(12)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
monthly["Expense"].plot(ax=ax, marker="o", color="#F43F5E", label="Expenses")
monthly["Income"].plot(ax=ax, marker="o", color="#10B981", label="Income")
ax.set_title("Monthly income and expenses")
ax.set_ylabel("Amount (INR)")
ax.legend()
fig.tight_layout()
plt.show()


## Monthly aggregate dataset used for forecasting

In [ ]:
monthly_dataset = pd.read_csv(PROJECT_ROOT / "datasets" / "monthly_expenses.csv")
print(f"Rows: {len(monthly_dataset)}")
monthly_dataset.tail()


In [ ]:
monthly_dataset[["total_expenses", "food_expenses", "transport_expenses", "shopping_expenses", "bills_expenses"]].describe().round(2)


## Budgets and insight templates

In [ ]:
budgets = pd.read_csv(PROJECT_ROOT / "datasets" / "budget_data.csv")
insights = pd.read_csv(PROJECT_ROOT / "datasets" / "financial_insights.csv")
print(f"Budget rows: {len(budgets)}")
print(f"Insight templates: {len(insights)}")
display(budgets.head())
display(insights.head())


## Takeaways

- Expenses span eight categories and roughly 20 months, which gives the
  classifier a stratified train/test split and the forecaster enough monthly
  history for a chronological hold-out.
- The transaction text is merchant/description based, which is exactly what the
  TF-IDF classifier sees in production.
- The monthly aggregate dataset is derived from the same spending patterns, so
  the forecasting notebook can be validated against a consistent history.
